# ColdSite-DTI cloud harness -- Wave A -- account ACC2

Generated by `notebooks/build_harness_notebook.py` from `config/waves.json`; do not hand-edit.

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** (the pre-flight refuses anything else) |
| Internet | **On** |
| Environment | **Pin to original** |
| How to run | **Save Version -> Save & Run All (Commit)** |
| First run | leave `DRY_RUN = True`: it does the pre-flight and prints the plan, then stops |

## What this account trains (12 cell(s))

| GPU queue | cell |
|---|---|
| gpu0 | davis_random_hyperattentiondti_seed5 |
| gpu0 | davis_random_moltrans_seed5 |
| gpu0 | davis_cold_drug_moltrans_seed5 |
| gpu0 | davis_random_coldsite_dti_seed5 |
| gpu0 | davis_cold_pair_moltrans_seed5 |
| gpu0 | davis_cold_pair_coldsite_dti_seed5 |
| gpu1 | davis_cold_target_hyperattentiondti_seed5 |
| gpu1 | davis_cold_drug_hyperattentiondti_seed5 |
| gpu1 | davis_cold_target_moltrans_seed5 |
| gpu1 | davis_cold_pair_hyperattentiondti_seed5 |
| gpu1 | davis_cold_drug_coldsite_dti_seed5 |
| gpu1 | davis_cold_target_coldsite_dti_seed5 |

Every cell of the plan appears once in `config/waves.json` across all accounts. Nothing here trains a
cell another account owns.

## The loop (each commit starts empty)

Kaggle output is per version. After a commit ends, download `results/` (the runner zips it, section 6),
add it as a private dataset, set `RESTORE_FROM = '/kaggle/input'` and run again. The runner copies back only
this account's cells, never over a file already there, verifies each finished cell's marker, and continues
partial cells from their last finished epoch. A commit self-stops before the session limit
(`config/harness.json`).

## 1. Settings

In [ ]:
ACCOUNT = 'ACC2'
WAVES = 'config/waves.json'
COMMIT = '526e9c3828a45f4ca55264278904241aac8dcd2d'        # the exact commit this notebook was generated for
REPO = 'https://github.com/Mahim56207/ColdSite-DTI_New.git'
DRY_RUN = True            # True: pre-flight + plan, then stop. False: train (spends GPU quota).
RESTORE_FROM = None       # None on the first commit; '/kaggle/input' after attaching the previous output

## 2. Session start and GPUs

In [ ]:
import os, subprocess, sys, time
START = time.time()          # the runner's self-stop is measured from here

print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout or 'nvidia-smi: none')

## 3. Clone the repo at the pinned commit

In [ ]:
WORK = '/kaggle/working'
SRC = f'{WORK}/ColdSite-DTI_New'
if not os.path.exists(SRC):
    subprocess.run(['git', 'clone', REPO, SRC], check=True)
os.chdir(SRC)
subprocess.run(['git', 'fetch', '--all', '-q'], check=True)
subprocess.run(['git', 'checkout', '-q', COMMIT], check=True)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert head == COMMIT, f'checked out {head}, expected {COMMIT}'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tabulate', 'subword-nmt'], check=True)
print('repo at', head)

## 4. Raw data and splits

The same steps every earlier notebook ran. The runner's pre-flight then checks the result against
`data/splits/MANIFEST.json` and `config/cloud_manifest.json` (SHA-256), which is stricter than the row counts
the older notebooks compared.

In [ ]:
BASE = 'https://raw.githubusercontent.com/hkmztrk/DeepDTA/master/data'
for ds in ('davis', 'kiba'):
    os.makedirs(f'src/data/baselines/deepdta/data/{ds}', exist_ok=True)
    for fname in ('ligands_can.txt', 'proteins.txt', 'Y'):
        target = f'src/data/baselines/deepdta/data/{ds}/{fname}'
        if not os.path.exists(target):
            subprocess.run(['curl', '-sL', f'{BASE}/{ds}/{fname}', '-o', target], check=True)
subprocess.run([sys.executable, '-m', 'src.data.load_data'], check=True)
subprocess.run([sys.executable, '-m', 'src.data.build_splits'], check=True)

## 5. Run: pre-flight, restore, one process per GPU

In [ ]:
RESULTS = f'{WORK}/results'
cmd = [sys.executable, '-u', '-m', 'src.cloud.runner', '--account', ACCOUNT, '--waves', WAVES,
       '--results-root', RESULTS, '--session-start', str(START)]
if RESTORE_FROM:
    cmd += ['--restore-from', RESTORE_FROM]
if DRY_RUN:
    cmd += ['--dry-run']
print(' '.join(cmd))
code_ = subprocess.run(cmd).returncode
print('runner exit code', code_, '(0 ok, 1 a cell needs attention, 2 refused by the pre-flight)')

## 6. What landed, and the file to take with you

In [ ]:
import glob, json
for path in sorted(glob.glob(f'{RESULTS}/runner_summary_*.json')):
    for row in json.load(open(path))['cells']:
        print(f"{row['outcome']:24s} {row['cell']}  {row['detail']}")
print()
print('complete-markers:', len(glob.glob(f'{RESULTS}/**/*_complete.json', recursive=True)))
ZIP = f'{ACCOUNT.lower()}_results.zip'
subprocess.run(f'cd {WORK} && rm -f {ZIP} && zip -qr {ZIP} results', shell=True)
print(f'{WORK}/{ZIP} -- download from the Output panel, add it to your restore dataset')